# Ultra Low Complexity Noise Suppression (paper-like ULCNet)

This notebook keeps the same overall flow as the current implementation, but the model and losses are moved much closer to the paper:

- **stage 1** is now a paper-style lightweight CRN with channelwise feature reorientation, depthwise-separable convolutions, a frequency-axis GRU, and subband temporal GRUs
- **stage 2** stays small and predicts a complex mask
- the notebook supports the two paper-style training settings:
  - **ULCNetFreq**: power-law compression + compressed-domain MSE
  - **ULCNetMS**: no compression + multi-resolution STFT loss

The code below is still kept practical for notebook-level experimentation and VoiceBank-DEMAND-style training.

## Install and import libraries

In [ ]:

# Install dependencies if needed
# !pip install torch torchaudio librosa soundfile tqdm pesq pystoi

import os
import glob
import time
import math
import numpy as np
import soundfile as sf
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from tqdm import tqdm

## GPU setup

In [ ]:

print("=" * 60)
print("GPU CONFIGURATION")
print("=" * 60)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory allocated: {torch.cuda.memory_allocated() / 1e9:.3f} GB")

## STFT helpers and power-law compression

Compared with the current notebook, the preprocessing stays intentionally close to the paper:
the real and imaginary parts are compressed separately before magnitude/phase features are built.

In [ ]:

def power_compress(x, alpha=0.3):
    """Sign-preserving power-law compression."""
    return torch.sign(x) * torch.abs(x).clamp_min(1e-12).pow(alpha)

def power_decompress(x, alpha=0.3):
    """Inverse of sign-preserving power-law compression."""
    return torch.sign(x) * torch.abs(x).clamp_min(1e-12).pow(1.0 / alpha)

def compute_stft(wav, n_fft=512, hop=256, win_length=512):
    """wav: [B, L] or [L]. returns complex STFT on the same device."""
    if wav.dim() == 1:
        wav = wav.unsqueeze(0)
    window = torch.hann_window(win_length, device=wav.device)
    return torch.stft(
        wav, n_fft=n_fft, hop_length=hop, win_length=win_length,
        window=window, return_complex=True, center=True
    )

def compute_istft(spec, n_fft=512, hop=256, win_length=512, length=None):
    """spec: [B, F, T] or [F, T]."""
    if spec.dim() == 2:
        spec = spec.unsqueeze(0)
    window = torch.hann_window(win_length, device=spec.device)
    return torch.istft(
        spec, n_fft=n_fft, hop_length=hop, win_length=win_length,
        window=window, length=length, center=True
    )

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## Paper-like building blocks

The biggest change versus the current notebook is in stage 1:

- the full frequency map is **not** flattened into one giant GRU
- frequency bins are first reoriented into subbands
- the convolution stack uses **depthwise-separable convolutions**
- a small **frequency-axis GRU** and **subband temporal GRUs** replace the oversized recurrent block

In [ ]:

class DepthwiseSeparableConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=(1, 3), padding=(0, 1), stride=(1, 1)):
        super().__init__()
        self.depthwise = nn.Conv2d(
            in_ch, in_ch, kernel_size=kernel_size, stride=stride,
            padding=padding, groups=in_ch, bias=False
        )
        self.pointwise = nn.Conv2d(in_ch, out_ch, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.PReLU(out_ch)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.act(x)
        return x

class ChannelwiseFeatureReorientation(nn.Module):
    """Approximation of the paper's channelwise feature reorientation."""
    def __init__(self, freq_bins=257, band_bins=48, band_hop=32):
        super().__init__()
        self.freq_bins = freq_bins
        self.band_bins = band_bins
        self.band_hop = band_hop
        # 8 bands of 48 bins with hop 32 -> 272 padded bins
        self.pad_to = band_bins + band_hop * 7

    def forward(self, x):
        # x: [B, T, F]
        b, t, f = x.shape
        if f < self.pad_to:
            x = F.pad(x, (0, self.pad_to - f))
        elif f > self.pad_to:
            x = x[..., :self.pad_to]
        bands = x.unfold(dimension=-1, size=self.band_bins, step=self.band_hop)
        # [B, T, 8, 48] -> [B, 8, T, 48]
        return bands.permute(0, 2, 1, 3).contiguous()

class FrequencyBiGRU(nn.Module):
    """BiGRU applied along the frequency axis for each time frame."""
    def __init__(self, in_features, hidden_size=64):
        super().__init__()
        self.gru = nn.GRU(
            input_size=in_features,
            hidden_size=hidden_size,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        # x: [B, C, T, F]
        b, c, t, f = x.shape
        x = x.permute(0, 2, 3, 1).contiguous().view(b * t, f, c)  # [B*T, F, C]
        y, _ = self.gru(x)
        y = y.view(b, t, f, -1).permute(0, 3, 1, 2).contiguous()   # [B, 2H, T, F]
        return y

class PaperLikeStage1(nn.Module):
    """Lightweight CRN-style magnitude mask estimator."""
    def __init__(self, freq_bins=257, band_bins=48, band_hop=32, num_bands=8):
        super().__init__()
        self.num_bands = num_bands
        self.reorient = ChannelwiseFeatureReorientation(freq_bins, band_bins, band_hop)

        # Paper-like depthwise-separable conv stack
        self.conv1 = DepthwiseSeparableConv2d(num_bands, 32)
        self.conv2 = DepthwiseSeparableConv2d(32, 64)
        self.conv3 = DepthwiseSeparableConv2d(64, 96)
        self.conv4 = DepthwiseSeparableConv2d(96, 128)

        # one gentle pooling step to keep the frequency resolution light
        self.pool = nn.MaxPool2d((1, 2), stride=(1, 2))

        # frequency-axis recurrent block
        self.fgru = FrequencyBiGRU(128, hidden_size=64)

        # pointwise projection after the recurrent block
        self.pw = nn.Conv2d(128, 64, kernel_size=1, bias=False)
        self.pw_bn = nn.BatchNorm2d(64)
        self.pw_act = nn.PReLU(64)

        # subband temporal GRU blocks (shared weights across subbands)
        self.band_gru = nn.GRU(
            input_size=64 * 3, hidden_size=64,
            num_layers=2, batch_first=True, bidirectional=True
        )

        # two FC layers for the intermediate magnitude mask
        self.fc1 = nn.Linear(num_bands * 128, 257)
        self.fc2 = nn.Linear(257, 257)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x: [B, T, F]
        x = self.reorient(x)                 # [B, 8, T, 48]
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.pool(self.conv3(x))         # [B, 96, T, 24]
        x = self.conv4(x)                    # [B, 128, T, 24]
        x = self.fgru(x)                     # [B, 128, T, 24]
        x = self.pw_act(self.pw_bn(self.pw(x)))  # [B, 64, T, 24]

        b, c, t, f = x.shape
        band_width = f // self.num_bands     # 24 -> 3
        band_feats = []

        for band_idx in range(self.num_bands):
            seg = x[:, :, :, band_idx * band_width:(band_idx + 1) * band_width]  # [B,64,T,3]
            seg = seg.permute(0, 2, 3, 1).contiguous().view(b, t, -1)             # [B,T,192]
            y, _ = self.band_gru(seg)                                             # [B,T,128]
            band_feats.append(y)

        feat = torch.cat(band_feats, dim=-1)  # [B, T, 1024]
        mask = self.sigmoid(self.fc2(F.relu(self.fc1(feat))))
        return mask

class PaperLikeStage2(nn.Module):
    """Tiny CNN used for the complex mask refinement stage."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=(1, 3), padding=(0, 1)),
            nn.ReLU(),
            nn.Conv2d(32, 2, kernel_size=1)
        )

    def forward(self, x):
        return self.net(x)

class ULCNetPaperLike(nn.Module):
    """End-to-end paper-like ULCNet implementation."""
    def __init__(self, freq_bins=257, alpha=0.3, use_compression=True):
        super().__init__()
        self.freq_bins = freq_bins
        self.alpha = alpha
        self.use_compression = use_compression
        self.stage1 = PaperLikeStage1(freq_bins=freq_bins)
        self.stage2 = PaperLikeStage2()

    def forward(self, mag, phase):
        # Stage 1: magnitude mask
        mag_mask = self.stage1(mag)  # [B, T, 257]

        # Stage 2: intermediate feature construction from noisy phase + stage-1 mask
        yr = mag_mask * torch.cos(phase)
        yi = mag_mask * torch.sin(phase)
        feat = torch.stack([yr, yi], dim=1)  # [B, 2, T, F]

        # complex mask refinement
        cmask = self.stage2(feat)
        mr = cmask[:, 0]
        mi = cmask[:, 1]
        return mag_mask, mr, mi

## Training losses

Two losses are provided, matching the paper's two training variants as closely as practical in a notebook:

- **compressed-domain MSE** for the compression-enabled model
- **multi-resolution STFT loss** for the no-compression variant

The multi-resolution loss is a practical stand-in for the paper's multi-scale objective.

In [ ]:

def compressed_domain_mse(est_r, est_i, target_r, target_i):
    return F.mse_loss(est_r, target_r) + F.mse_loss(est_i, target_i)

def multi_resolution_stft_loss(pred, target, fft_sizes=(256, 512, 1024)):
    """A practical multi-scale spectral loss."""
    if pred.dim() == 1:
        pred = pred.unsqueeze(0)
    if target.dim() == 1:
        target = target.unsqueeze(0)

    total = 0.0
    for n_fft in fft_sizes:
        hop = n_fft // 4
        win = n_fft
        window = torch.hann_window(win, device=pred.device)

        p = torch.stft(pred, n_fft=n_fft, hop_length=hop, win_length=win,
                       window=window, return_complex=True, center=True)
        t = torch.stft(target, n_fft=n_fft, hop_length=hop, win_length=win,
                       window=window, return_complex=True, center=True)

        p_mag = torch.abs(p).clamp_min(1e-7)
        t_mag = torch.abs(t).clamp_min(1e-7)

        sc_num = torch.linalg.norm(t_mag - p_mag)
        sc_den = torch.linalg.norm(t_mag).clamp_min(1e-7)
        spectral_convergence = sc_num / sc_den
        log_mag = F.l1_loss(torch.log(p_mag), torch.log(t_mag))
        total = total + spectral_convergence + log_mag

    return total / len(fft_sizes)

## Dataset loader and collate function

This keeps the same notebook-style workflow as the current implementation, but the dataset is now expected to come from
VoiceBank-DEMAND-style paired clean/noisy folders.

In [ ]:

from torch.nn.utils.rnn import pad_sequence

def collate_fn(batch):
    mixes = [b[0].float() for b in batch]
    cleans = [b[1].float() for b in batch]
    lengths = torch.tensor([m.shape[0] for m in mixes], dtype=torch.long)
    mix_p = pad_sequence(mixes, batch_first=True)
    clean_p = pad_sequence(cleans, batch_first=True)
    return mix_p, clean_p, lengths

def list_wavs(root):
    return sorted(glob.glob(os.path.join(root, "*.wav")))

def pair_by_basename(noisy_dir, clean_dir):
    noisy_files = {os.path.basename(p): p for p in list_wavs(noisy_dir)}
    clean_files = {os.path.basename(p): p for p in list_wavs(clean_dir)}
    common = sorted(set(noisy_files) & set(clean_files))
    noisy_paths = [noisy_files[k] for k in common]
    clean_paths = [clean_files[k] for k in common]
    return noisy_paths, clean_paths

class PairedSpeechDataset(torch.utils.data.Dataset):
    def __init__(self, noisy_files, clean_files, sr=16000):
        assert len(noisy_files) == len(clean_files)
        self.noisy = noisy_files
        self.clean = clean_files
        self.sr = sr

    def __len__(self):
        return len(self.noisy)

    def __getitem__(self, idx):
        noisy, _ = librosa.load(self.noisy[idx], sr=self.sr)
        clean, _ = librosa.load(self.clean[idx], sr=self.sr)
        L = min(len(noisy), len(clean))
        noisy = noisy[:L]
        clean = clean[:L]
        return torch.tensor(noisy, dtype=torch.float32), torch.tensor(clean, dtype=torch.float32)

## Training and inference

The training loop mirrors the current notebook structure, but the internal forward pass is now closer to the paper.
You can switch between the two paper-style settings with:
- `loss_type="freq_mse"` for the compression-enabled model
- `loss_type="ms"` for the no-compression model

In [ ]:

def enhance(model, wav, device, n_fft=512, hop=256):
    """End-to-end enhancement for a single waveform."""
    model.eval()
    if isinstance(wav, np.ndarray):
        wav = torch.tensor(wav, dtype=torch.float32)
    wav = wav.to(device)
    orig_len = wav.shape[0]
    wav_b = wav.unsqueeze(0)

    with torch.no_grad():
        X = compute_stft(wav_b, n_fft=n_fft, hop=hop, win_length=n_fft)
        if model.use_compression:
            Xr = power_compress(X.real, alpha=model.alpha)
            Xi = power_compress(X.imag, alpha=model.alpha)
        else:
            Xr = X.real
            Xi = X.imag

        mag = torch.sqrt(Xr**2 + Xi**2).permute(0, 2, 1)   # [B, T, F]
        phase = torch.atan2(Xi, Xr).permute(0, 2, 1)       # [B, T, F]

        mag_mask, mr, mi = model(mag, phase)
        mr = mr.permute(0, 2, 1)
        mi = mi.permute(0, 2, 1)

        est_r_c = mr * Xr - mi * Xi
        est_i_c = mr * Xi + mi * Xr

        if model.use_compression:
            est_r = power_decompress(est_r_c, alpha=model.alpha)
            est_i = power_decompress(est_i_c, alpha=model.alpha)
        else:
            est_r = est_r_c
            est_i = est_i_c

        spec = torch.complex(est_r, est_i)
        out = compute_istft(spec, n_fft=n_fft, hop=hop, win_length=n_fft, length=orig_len)
        return out.squeeze(0).cpu().numpy()

def train_one_epoch(model, loader, optimizer, device, loss_type="freq_mse", n_fft=512, hop=256):
    model.train()
    total_loss = 0.0
    total_batches = 0

    for mix, clean, lengths in tqdm(loader):
        mix = mix.to(device)
        clean = clean.to(device)
        lengths = lengths.to(device)

        X = compute_stft(mix, n_fft=n_fft, hop=hop, win_length=n_fft)

        if model.use_compression:
            Xr = power_compress(X.real, alpha=model.alpha)
            Xi = power_compress(X.imag, alpha=model.alpha)
        else:
            Xr = X.real
            Xi = X.imag

        mag = torch.sqrt(Xr**2 + Xi**2).permute(0, 2, 1)
        phase = torch.atan2(Xi, Xr).permute(0, 2, 1)

        mag_mask, mr, mi = model(mag, phase)
        mr = mr.permute(0, 2, 1)
        mi = mi.permute(0, 2, 1)

        est_r_c = mr * Xr - mi * Xi
        est_i_c = mr * Xi + mi * Xr

        if loss_type == "freq_mse":
            clean_X = compute_stft(clean, n_fft=n_fft, hop=hop, win_length=n_fft)
            if model.use_compression:
                target_r = power_compress(clean_X.real, alpha=model.alpha)
                target_i = power_compress(clean_X.imag, alpha=model.alpha)
            else:
                target_r = clean_X.real
                target_i = clean_X.imag
            loss = compressed_domain_mse(est_r_c, est_i_c, target_r, target_i)

        elif loss_type == "ms":
            if model.use_compression:
                est_r = power_decompress(est_r_c, alpha=model.alpha)
                est_i = power_decompress(est_i_c, alpha=model.alpha)
            else:
                est_r = est_r_c
                est_i = est_i_c
            spec = torch.complex(est_r, est_i)
            out = compute_istft(spec, n_fft=n_fft, hop=hop, win_length=n_fft, length=mix.shape[1])

            losses = []
            for i, l in enumerate(lengths):
                li = int(l.item())
                if li > 0:
                    losses.append(multi_resolution_stft_loss(out[i, :li], clean[i, :li]))
            loss = torch.stack(losses).mean() if losses else torch.tensor(0.0, device=device)
        else:
            raise ValueError("loss_type must be 'freq_mse' or 'ms'.")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_batches += 1

    return total_loss / max(total_batches, 1)

def compute_test_mse(model, noisy_files, clean_files, device, n_fft=512, hop=256):
    model.eval()
    total_mse = 0.0
    total_samples = 0

    with torch.no_grad():
        for nfile, cfile in zip(noisy_files, clean_files):
            noisy_wav, _ = librosa.load(nfile, sr=16000)
            clean_wav, _ = librosa.load(cfile, sr=16000)

            enhanced_wav = enhance(model, noisy_wav, device, n_fft=n_fft, hop=hop)
            L = min(len(enhanced_wav), len(clean_wav))
            mse = np.mean((enhanced_wav[:L] - clean_wav[:L]) ** 2)

            total_mse += mse * L
            total_samples += L

    return total_mse / max(total_samples, 1)

def save_model(model, path):
    torch.save({
        "state_dict": model.state_dict(),
        "freq_bins": model.freq_bins,
        "alpha": model.alpha,
        "use_compression": model.use_compression,
    }, path)

def load_model(path, device):
    ckpt = torch.load(path, map_location=device)
    model = ULCNetPaperLike(
        freq_bins=ckpt.get("freq_bins", 257),
        alpha=ckpt.get("alpha", 0.3),
        use_compression=ckpt.get("use_compression", True),
    ).to(device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    return model

## Example setup

Replace these paths with your VoiceBank-DEMAND locations.

The same notebook can be used for:
- **ULCNetFreq**: `alpha=0.3`, `use_compression=True`, `loss_type="freq_mse"`
- **ULCNetMS**: `alpha=1.0`, `use_compression=False`, `loss_type="ms"`

In [ ]:

# Example paths — update for your environment
train_noisy_dir = "/path/to/VoiceBank-DEMAND/noisy_trainset_wav"
train_clean_dir = "/path/to/VoiceBank-DEMAND/clean_trainset_wav"
test_noisy_dir = "/path/to/VoiceBank-DEMAND/noisy_testset_wav"
test_clean_dir = "/path/to/VoiceBank-DEMAND/clean_testset_wav"

# Pair files by basename
train_noisy_files, train_clean_files = pair_by_basename(train_noisy_dir, train_clean_dir)
test_noisy_files, test_clean_files = pair_by_basename(test_noisy_dir, test_clean_dir)

print(f"Train pairs: {len(train_noisy_files)}")
print(f"Test pairs:  {len(test_noisy_files)}")

## Model creation and parameter count

This is the point where the new implementation differs most from the earlier notebook:
the parameter count is now in the same ballpark as the paper, instead of being dominated by one huge flattened GRU.

In [ ]:

model = ULCNetPaperLike(freq_bins=257, alpha=0.3, use_compression=True).to(device)
print(model)
print(f"Trainable parameters: {count_parameters(model) / 1e6:.3f} M")

## Training example

Use the frequency-domain MSE objective for the compression-enabled variant,
or the multi-resolution STFT loss for the no-compression variant.

In [ ]:

# Example DataLoader
# train_dataset = PairedSpeechDataset(train_noisy_files, train_clean_files, sr=16000)
# train_loader = torch.utils.data.DataLoader(
#     train_dataset, batch_size=4, shuffle=True, collate_fn=collate_fn, num_workers=0
# )

# Example optimizer
# optimizer = torch.optim.Adam(model.parameters(), lr=4e-4)

# Example training loop
# for epoch in range(10):
#     start = time.time()
#     loss = train_one_epoch(model, train_loader, optimizer, device, loss_type="freq_mse")
#     print(f"Epoch {epoch+1:02d} | Loss: {loss:.6f} | Time: {time.time() - start:.2f}s")
#     save_model(model, f"ulcnet_paperlike_epoch_{epoch+1}.pth")

## Test-set evaluation example

This keeps the same style as the current notebook, while letting you compare the paper-like variants more cleanly.

In [ ]:

# Example:
# mse = compute_test_mse(model, test_noisy_files, test_clean_files, device)
# print("Test MSE:", mse)

## Notes

- The notebook is intentionally still easy to follow and modify.
- The most important architectural difference versus the current notebook is the replacement of the giant flattened GRU with a much smaller subband-aware recurrent stack.
- For the final project, you can keep this notebook as the paper-like model branch and compare it against the classical STFT-based baseline in the Gradio app.